# Step 2 — SECOM preprocessing and failure-detection baseline

**Goal:** choose a small set of sensor inputs, reproducibly and without learning from validation/test rows. Run `scripts/02_prepare_features.py` first.

Connection to ECE 8803: Module 2 high-dimensional FDC, Module 6 generalization/data leakage, Module 7 rare-defect screening and escape-vs-overkill trade-offs.

## 1. Load results and verify the holdout split

Original label `-1` is pass and `+1` fail; project label `0` is pass and `1` fail. The saved split uses seed 42. **Do not train on validation/test rows.**

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "reports" / "split_summary.json").exists():
    ROOT = ROOT.parent
assert (ROOT / "reports" / "split_summary.json").exists(), "Run scripts/02_prepare_features.py first."

summary = json.loads((ROOT / "reports" / "split_summary.json").read_text(encoding="utf-8"))
display(pd.DataFrame({name: summary[name] for name in ("train", "validation", "test")}).T)


## 2. Inspect our chosen hardware-sized feature sets

ANOVA F-scores are calculated with **training labels only**; sensor IDs are anonymous. A high score indicates statistical association, not physical causation.

In [ ]:
selected = pd.read_csv(ROOT / "reports" / "selected_features.csv")
for k in (8, 16, 32):
    print(f"\n{k} features (hardware input order):")
    display(selected.loc[selected.feature_count == k, ["input_position", "sensor", "anova_f_score_train", "missing_percent_train"]].reset_index(drop=True))


## 3. Validation-only benchmark

The always-pass classifier is expected to be very accurate in this imbalanced dataset but have failure recall 0. The logistic regression model uses `class_weight="balanced"` to pay more attention to failing examples. The classification threshold is fixed at 0.5 for this initial comparison.

In [ ]:
metrics = pd.read_csv(ROOT / "reports" / "validation_baseline_metrics.csv")
display(metrics[["model", "features", "accuracy", "failure_recall", "false_positive_rate", "pr_auc_average_precision", "escaped_failure_fn", "false_alarm_fp"]].round(3))


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.8))
for _, row in metrics.iterrows():
    ax.scatter(row.false_positive_rate, row.failure_recall, s=85)
    ax.annotate(row["model"], (row.false_positive_rate, row.failure_recall), xytext=(6,5), textcoords="offset points")
ax.set_xlabel("False-positive rate: passing samples incorrectly flagged")
ax.set_ylabel("Recall: fraction of failing samples caught")
ax.set_title("SECOM validation — model screening trade-off")
ax.set_xlim(-.04, 1.02)
ax.set_ylim(-.04, 1.02)
ax.grid(alpha=.25)
plt.show()


## 4. Interpretation and next experiment

- **False negative / escaped failure:** model says pass while the ground-truth label is fail.
- **False positive / false alarm or overkill:** model says fail while the label is pass.
- **PR-AUC (average precision)** is important when positive failures are rare.
- A random stratified split is an initial estimate, not a demonstration of resilience to chronological process drift. SECOM timestamps enable a later forward-in-time experiment.
- We have *not* tested actual process drift, built an SPC chart, or determined physical sensor causes.
- **Hardware relevance:** model input count k maps to k weighted terms; actual silicon area, timing, and power require synthesis.

**Next:** compare random forest and a small neural network, tune failure/false-alarm trade-offs only using validation, and finally quantize one model for SystemVerilog. The test set remains untouched until the end.